In [1]:
import yfinance as yf

# Le digo qué acción quiero — usamos Bancolombia
ticker = yf.Ticker("CIB")  # CIB es Bancolombia en bolsa de NY

# Le pido la información básica del instrumento
info = ticker.info

print("Nombre:", info["longName"])
print("Mercado:", info["exchange"])
print("Moneda:", info["currency"])
print("Precio actual:", info["currentPrice"])

Nombre: Grupo Cibest S.A.
Mercado: NYQ
Moneda: USD
Precio actual: 74.9


In [2]:
import yfinance as yf
import pandas as pd

ticker = yf.Ticker("CIB")

# Pido precios de los últimos 30 días
precios = ticker.history(period="30d")

print(precios.head())

                                Open       High        Low      Close  Volume  \
Date                                                                            
2026-04-28 00:00:00-04:00  69.669998  69.690002  67.320000  68.699997  494100   
2026-04-29 00:00:00-04:00  68.339996  68.989998  66.410004  67.290001  480200   
2026-04-30 00:00:00-04:00  67.860001  68.379997  66.589996  68.190002  348700   
2026-05-01 00:00:00-04:00  68.080002  68.160004  66.379997  67.040001  282900   
2026-05-04 00:00:00-04:00  67.120003  67.589996  64.699997  65.220001  243500   

                           Dividends  Stock Splits  
Date                                                
2026-04-28 00:00:00-04:00        0.0           0.0  
2026-04-29 00:00:00-04:00        0.0           0.0  
2026-04-30 00:00:00-04:00        0.0           0.0  
2026-05-01 00:00:00-04:00        0.0           0.0  
2026-05-04 00:00:00-04:00        0.0           0.0  


In [8]:
import subprocess
subprocess.run([
    r"C:\Users\busta\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe",
    "-m", "pip", "install", "pyarrow"
])

CompletedProcess(args=['C:\\Users\\busta\\AppData\\Local\\Microsoft\\WindowsApps\\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\\python.exe', '-m', 'pip', 'install', 'pyarrow'], returncode=0)

In [3]:
import yfinance as yf
import pandas as pd
import os
from datetime import datetime

# --- EXTRACT ---
print("Extrayendo datos de Yahoo Finance...")
ticker = yf.Ticker("CIB")
precios = ticker.history(period="30d")

# --- LOAD → BRONZE ---
# Creamos la carpeta si no existe
os.makedirs("bronze", exist_ok=True)

# Nombre del archivo con la fecha de hoy
fecha_hoy = datetime.today().strftime("%Y_%m_%d")
ruta_bronze = f"bronze/precios_CIB_{fecha_hoy}.parquet"

# Guardamos tal como llegó — sin tocar nada
precios.to_parquet(ruta_bronze)

print(f"✅ Guardado en Bronze: {ruta_bronze}")
print(f"   {len(precios)} filas, {len(precios.columns)} columnas")

Extrayendo datos de Yahoo Finance...
✅ Guardado en Bronze: bronze/precios_CIB_2026_06_09.parquet
   30 filas, 7 columnas


In [4]:
# --- TRANSFORM → SILVER ---
os.makedirs("silver", exist_ok=True)

df_silver = precios.copy()

# Limpiar: quedarnos solo con columnas útiles
df_silver = df_silver[["Open", "High", "Low", "Close", "Volume"]]

# Renombrar a español para el negocio
df_silver.columns = ["apertura", "maximo", "minimo", "cierre", "volumen"]

# Agregar columna de ticker
df_silver["ticker"] = "CIB"

# Quitar filas con nulos
df_silver = df_silver.dropna()

# Guardar Silver
ruta_silver = f"silver/precios_CIB_{fecha_hoy}.parquet"
df_silver.to_parquet(ruta_silver)
print(f"✅ Silver listo: {ruta_silver}")

# --- TRANSFORM → GOLD ---
os.makedirs("gold", exist_ok=True)

# Resumen para negocio: estadísticas del período
df_gold = df_silver.agg({
    "cierre":  ["mean", "min", "max"],
    "volumen": ["mean", "sum"]
}).round(2)

# Guardar Gold
ruta_gold = f"gold/resumen_CIB_{fecha_hoy}.parquet"
df_gold.to_parquet(ruta_gold)
print(f"✅ Gold listo: {ruta_gold}")
print("\nResumen del período:")
print(df_gold)

✅ Silver listo: silver/precios_CIB_2026_06_09.parquet
✅ Gold listo: gold/resumen_CIB_2026_06_09.parquet

Resumen del período:
      cierre      volumen
mean   67.81    393207.07
min    63.16          NaN
max    74.90          NaN
sum      NaN  11796212.00
